# IAIP — Indian Premier League Analysis & Insights Project
**Task 2 | EDA on Sports Data | Tool: Python**

This notebook performs a complete Exploratory Data Analysis on the IPL dataset:
- 756 matches across 12 seasons (2008–2019)
- 179,078 ball-by-ball deliveries
- Team success, player performance, and endorsement recommendations


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (10, 5)})

matches = pd.read_csv('matches.csv')
deliveries = pd.read_csv('deliveries.csv')

print(f"Matches shape   : {matches.shape}")
print(f"Deliveries shape: {deliveries.shape}")
print(f"Seasons         : {sorted(matches['season'].unique())}")


## 1. Dataset Overview

In [ ]:
print("=== MATCHES DATASET ===")
print(matches.info())
print("\nSample rows:")
matches.head(3)


In [ ]:
print("=== DELIVERIES DATASET ===")
print(deliveries.info())
deliveries.head(3)


In [ ]:
print("Missing values in matches:")
print(matches.isnull().sum()[matches.isnull().sum() > 0])
print("\nMissing values in deliveries:")
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])


## 2. Season-wise Match Distribution

In [ ]:
season_matches = matches['season'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(season_matches.index, season_matches.values, marker='o', color='#1a73e8', linewidth=2.5, markersize=8)
ax.fill_between(season_matches.index, season_matches.values, alpha=0.15, color='#1a73e8')
ax.set_title('IPL Matches Per Season (2008–2019)', fontsize=14, fontweight='bold')
ax.set_xlabel('Season'); ax.set_ylabel('Matches')
ax.set_xticks(season_matches.index)
for x, y in zip(season_matches.index, season_matches.values):
    ax.text(x, y+0.8, str(y), ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 3. Team Success Analysis

In [ ]:
team_wins = matches['winner'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1a73e8' if i < 2 else '#4a90d9' if i < 5 else '#82b4e8' for i in range(len(team_wins))]
ax.barh(team_wins.index[::-1], team_wins.values[::-1], color=colors[::-1], edgecolor='white')
ax.set_title('IPL All-Time Team Wins (Top 10)', fontsize=14, fontweight='bold')
ax.set_xlabel('Wins')
for i, val in enumerate(team_wins.values[::-1]):
    ax.text(val + 1, i, str(val), va='center', fontsize=10)
ax.set_xlim(0, team_wins.max() + 15)
plt.tight_layout()
plt.show()


In [ ]:
all_teams = pd.concat([matches['team1'], matches['team2']]).value_counts()
win_pct = (matches['winner'].value_counts() / all_teams * 100).dropna().round(1).sort_values(ascending=False)
print("Win Percentage by Team:")
print(win_pct.to_frame('Win %').head(12).to_string())


In [ ]:
toss_win = matches[matches['toss_winner'] == matches['winner']].shape[0]
bat2nd_wins = (matches['win_by_wickets'] > 0).sum()
bat1st_wins = (matches['win_by_runs'] > 0).sum()

print(f"Toss to match win conversion : {round(toss_win/len(matches)*100, 1)}%")
print(f"Won batting 2nd (by wickets) : {bat2nd_wins} ({round(bat2nd_wins/(bat1st_wins+bat2nd_wins)*100,1)}%)")
print(f"Won batting 1st (by runs)    : {bat1st_wins} ({round(bat1st_wins/(bat1st_wins+bat2nd_wins)*100,1)}%)")

fig, ax = plt.subplots(figsize=(6, 5))
ax.pie([bat2nd_wins, bat1st_wins],
       labels=['Batting 2nd\n(by wickets)', 'Batting 1st\n(by runs)'],
       colors=['#1a73e8', '#e8604a'], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2}, textprops={'fontsize': 11})
ax.set_title('How IPL Matches Are Won', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 4. Top Batsmen

In [ ]:
total_runs = deliveries.groupby('batsman')['batsman_runs'].sum().sort_values(ascending=False)
top10_bat = total_runs.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(top10_bat)), top10_bat.values, color='#4a90d9', edgecolor='white', width=0.65)
ax.set_xticks(range(len(top10_bat)))
ax.set_xticklabels(top10_bat.index, rotation=30, ha='right')
ax.set_ylabel('Total Runs'); ax.set_title('Top 10 IPL Run Scorers (All-Time)', fontsize=14, fontweight='bold')
for i, val in enumerate(top10_bat.values):
    ax.text(i, val + 30, f'{val:,}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
balls_faced = deliveries[deliveries['wide_runs'] == 0].groupby('batsman').size()
strike_rate = (total_runs / balls_faced * 100).dropna()
sr_qual = strike_rate[total_runs >= 500].sort_values(ascending=False).head(10).round(2)
print("Strike Rate Leaders (min 500 runs):")
print(sr_qual.to_frame('Strike Rate').to_string())


In [ ]:
pom = matches['player_of_match'].value_counts().head(10)
print("Player of the Match Awards:")
print(pom.to_frame('Awards').to_string())


## 5. Top Bowlers

In [ ]:
non_runout = ['run out', 'retired hurt', 'obstructed the field']
wickets_df = deliveries[deliveries['dismissal_kind'].notna() &
                        ~deliveries['dismissal_kind'].isin(non_runout)]
top_bowlers = wickets_df.groupby('bowler').size().sort_values(ascending=False)
top10_bowl = top_bowlers.head(10)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(top10_bowl)), top10_bowl.values, color='#e8604a', edgecolor='white', width=0.65)
ax.set_xticks(range(len(top10_bowl)))
ax.set_xticklabels(top10_bowl.index, rotation=30, ha='right')
ax.set_ylabel('Wickets'); ax.set_title('Top 10 IPL Wicket Takers (All-Time)', fontsize=14, fontweight='bold')
for i, val in enumerate(top10_bowl.values):
    ax.text(i, val + 1, str(val), ha='center', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
total_balls = deliveries.groupby('bowler').size()
total_runs_b = deliveries.groupby('bowler')['total_runs'].sum()
economy = (total_runs_b / total_balls * 6).round(2)
eco_qual = economy[total_balls >= 240].sort_values().head(10)
print("Best Economy Rates (min 240 balls):")
print(eco_qual.to_frame('Economy').to_string())


## 6. Endorsement Recommendations

In [ ]:
print("""
╔══════════════════════════════════════════════════════════╗
║         ENDORSEMENT RECOMMENDATIONS — IAIP              ║
╠══════════════════════════════════════════════════════════╣
║ TIER 1 — MARQUEE                                        ║
║  Virat Kohli    : 5,434 runs | Mass market, FMCG       ║
║  MS Dhoni       : 17 POM | Trust, insurance, auto      ║
║  AB de Villiers : 20 POM | Lifestyle, tech, sports     ║
╠══════════════════════════════════════════════════════════╣
║ TIER 2 — HIGH VALUE                                     ║
║  Chris Gayle    : 21 POM | Energy, youth, streetwear   ║
║  Rohit Sharma   : 4,914 runs | Premium, banking        ║
║  Suresh Raina   : 5,415 runs | North India reach       ║
╠══════════════════════════════════════════════════════════╣
║ TIER 3 — TEAMS                                          ║
║  Mumbai Indians  : 109 wins | Widest fan base          ║
║  Chennai Super Kings: 61% win rate | Loyalty brands    ║
╚══════════════════════════════════════════════════════════╝
""")


## 7. Key Insights Summary

| Insight | Finding |
|---------|---------|
| Most successful team | Mumbai Indians (109 wins) |
| Highest win rate | Chennai Super Kings (61%) |
| Top run scorer | Virat Kohli (5,434 runs) |
| Top wicket taker | Lasith Malinga (170 wickets) |
| Best match winners | Chris Gayle (21 POM awards) |
| Toss advantage | 52% — marginal but real |
| Batting strategy | Field first — teams bat 2nd win 55% |
| Best endorsement pick | Virat Kohli (runs + reach + recognition) |
